# Project: Building a Hybrid Search Engine

Build a hybrid system that combines dense and sparse vectors with Reciprocal Rank Fusion, demonstrating how to get the best of both semantic understanding and keyword precision.

## Mission

Create a production-ready hybrid search system that leverages both dense and sparse vectors to deliver superior search results. You’ll implement the complete hybrid pipeline and compare its performance against single-vector approaches.

## What to Build

A hybrid search system that demonstrates:

- Dense vector search for semantic understanding
- Sparse vector search for exact keyword matching
- Reciprocal Rank Fusion to combine results intelligently
- Performance comparison between hybrid and single-vector approaches
- Domain optimization for your specific use case

## Step 1: Set Up Hybrid Collection

In [2]:
pip install --upgrade ipywidgets

   ---------------------------------------- 0.0/139.8 kB ? eta -:--:--
   -- ------------------------------------- 10.2/139.8 kB ? eta -:--:--
   -------- ------------------------------ 30.7/139.8 kB 262.6 kB/s eta 0:00:01
   ----------- --------------------------- 41.0/139.8 kB 281.8 kB/s eta 0:00:01
   --------------------------- ---------- 102.4/139.8 kB 535.8 kB/s eta 0:00:01
   -------------------------------------- 139.8/139.8 kB 638.3 kB/s eta 0:00:00
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   - -------------------------------------- 30.7/914.9 kB ? eta -:--:--
   ----------- ---------------------------- 256.0/914.9 kB 3.1 MB/s eta 0:00:01
   ----------------- ---------------------- 389.1/914.9 kB 3.0 MB/s eta 0:00:01
   ------------------------ --------------- 553.0/914.9 kB 3.5 MB/s eta 0:00:01
   ------------------------------- -------- 716.8/914.9 kB 3.5 MB/s eta 0:00:01
   ------------------------------------- -- 849.9/914.9 kB 3.4 MB/s eta 0:


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
import time
import os

client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))

collection_name = "day3_hybrid_search"

# Create hybrid collection
client.create_collection(
    collection_name=collection_name,
    vectors_config={
        "dense": models.VectorParams(size=384, distance=models.Distance.COSINE)
    },
    sparse_vectors_config={
        "sparse": models.SparseVectorParams(
            index=models.SparseIndexParams(on_disk=False)
        )
    }
)

True

## Prepare Your Dataset

In [5]:
# Example: Recipe collection
my_dataset = [
    {
        "title": "Classic Beef Bourguignon",
        "description": """A rich, wine-braised beef stew from Burgundy, France.
        Tender chunks of beef are slowly simmered with pearl onions, mushrooms,
        and bacon in a deep red wine sauce. The long, slow cooking process
        develops complex flavors and creates a luxurious, velvety texture.
        Perfect for cold winter evenings when you want something hearty and
        comforting. Traditionally served with crusty bread or creamy mashed
        potatoes to soak up the incredible sauce.""",
        "cuisine": "French",
        "difficulty": "Intermediate",
        "time": "3 hours"
    },
    {
        "title": "Thai Green Curry with Chicken",
        "description": """An aromatic and vibrant curry from Thailand featuring tender
        chicken in a coconut milk base infused with fragrant green curry paste, Thai basil,
        and kaffir lime leaves. The sauce balances spicy, sweet, and savory notes with
        bamboo shoots, eggplant, and bell peppers adding texture. Quick to prepare yet
        bursting with complex flavors, this dish brings the authentic taste of Bangkok
        street food to your kitchen. Serve over jasmine rice to soak up every drop of
        the creamy, spice-laden sauce.""",
        "cuisine": "Thai",
        "difficulty": "Easy",
        "time": "30 minutes"
    },
    {
        "title": "Homemade Margherita Pizza",
        "description": """The quintessential Neapolitan pizza that celebrates simplicity
        and quality ingredients. A thin, chewy crust with charred bubbles holds a bright
        San Marzano tomato sauce, creamy fresh mozzarella, and fragrant basil leaves.
        The key is high heat and minimal toppings, allowing each element to shine.
        Making the dough from scratch requires patience as it slowly ferments, developing
        complex flavors and that signature airy texture. Perfect for weekend cooking
        when you want to impress with authentic Italian technique.""",
        "cuisine": "Italian",
        "difficulty": "Intermediate",
        "time": "2 hours (plus dough rise)"
    },
    {
        "title": "Miso-Glazed Salmon with Sesame Vegetables",
        "description": """A modern Japanese-inspired dish that's both elegant and nutritious.
        Fresh salmon fillets are marinated in a sweet-savory miso glaze with mirin and sake,
        then broiled until caramelized and slightly charred at the edges. The umami-rich
        glaze creates a beautiful lacquered finish. Paired with crisp stir-fried vegetables
        tossed in sesame oil and garnished with toasted sesame seeds. This restaurant-quality
        meal comes together in under 25 minutes, making it perfect for busy weeknights when
        you don't want to sacrifice flavor or presentation.""",
        "cuisine": "Japanese",
        "difficulty": "Easy",
        "time": "25 minutes"
    },
    {
        "title": "Moroccan Lamb Tagine with Apricots",
        "description": """A fragrant North African stew that marries tender lamb with sweet
        dried apricots, aromatic spices, and preserved lemons. Slow-cooked in a traditional
        cone-shaped tagine or heavy pot, the meat becomes fall-apart tender while absorbing
        the warm spices of cinnamon, cumin, and ginger. Chickpeas add heartiness, while
        honey and apricots provide a delicate sweetness that balances the savory depth.
        The long, gentle cooking process allows the complex spice blend to fully develop.
        Serve over fluffy couscous with fresh cilantro and toasted almonds for a truly
        exotic dining experience.""",
        "cuisine": "Moroccan",
        "difficulty": "Intermediate",
        "time": "2.5 hours"
    },
    {
        "title": "Classic Chicken Caesar Salad",
        "description": """A timeless American restaurant staple that's surprisingly easy to
        master at home. Crisp romaine lettuce is tossed with a bold, creamy dressing made
        from anchovies, garlic, lemon, egg yolk, and Parmesan cheese. Topped with juicy
        grilled chicken breast, crunchy house-made croutons, and extra shaved Parmesan.
        The key is the dressing—punchy, garlicky, and perfectly emulsified. While often
        considered simple, a well-executed Caesar showcases the power of balancing strong,
        complementary flavors. Great for a light lunch or dinner that feels both indulgent
        and refreshing.""",
        "cuisine": "American",
        "difficulty": "Easy",
        "time": "20 minutes"
    },
    {
        "title": "Indian Butter Chicken (Murgh Makhani)",
        "description": """A beloved North Indian classic featuring tender chicken in a
        luxuriously creamy tomato-based sauce. The chicken is first marinated in yogurt
        and spices, then grilled or pan-fried for a smoky char before being simmered in
        a velvety sauce enriched with butter, cream, and aromatic spices like garam masala,
        fenugreek, and cardamom. The result is a perfect balance of tangy, sweet, and
        mildly spiced flavors with a silky texture. This restaurant favorite is easier to
        make at home than you'd think, and the aroma while cooking will transport you to
        the bustling streets of Delhi. Serve with naan bread and basmati rice.""",
        "cuisine": "Indian",
        "difficulty": "Intermediate",
        "time": "1 hour"
    },
    {
        "title": "Spanish Paella Valenciana",
        "description": """The iconic rice dish from Valencia that's meant for sharing and
        celebrating. Saffron-infused short-grain rice is cooked with chicken, rabbit, and
        green beans in a wide, shallow pan until it develops the prized 'socarrat'—a
        crispy, caramelized bottom layer. Fresh rosemary and smoked paprika add depth,
        while the saffron lends its distinctive golden color and earthy aroma. Traditionally
        cooked over an open fire, this one-pan feast brings people together. Making paella
        is as much about the ritual and patience as it is about the ingredients. The key
        is resisting the urge to stir, allowing those delicious crispy bits to form.""",
        "cuisine": "Spanish",
        "difficulty": "Advanced",
        "time": "1.5 hours"
    },
    {
        "title": "Vietnamese Pho Bo (Beef Noodle Soup)",
        "description": """Vietnam's national dish—a deeply aromatic beef broth that takes
        hours to perfect. Beef bones are simmered with charred onions, ginger, star anise,
        cinnamon, and coriander seeds until the broth becomes rich, clear, and intensely
        flavorful. Served over silky rice noodles with thinly sliced rare beef that cooks
        in the steaming broth, then finished with fresh herbs, lime, jalapeños, and bean
        sprouts. Each bowl is customizable at the table, making it interactive and personal.
        While time-intensive, the reward is a soul-warming bowl of pure comfort that rivals
        any pho shop in Hanoi.""",
        "cuisine": "Vietnamese",
        "difficulty": "Advanced",
        "time": "4 hours"
    },
    {
        "title": "Greek Moussaka",
        "description": """A hearty, layered casserole that's Greece's answer to lasagna.
        Tender slices of eggplant and potato are layered with a rich, spiced ground lamb
        and tomato sauce flavored with cinnamon, oregano, and a hint of red wine. The
        crown jewel is the creamy béchamel sauce on top, which bakes to a golden brown.
        Each forkful delivers multiple textures and layers of Mediterranean flavor. While
        it requires some prep work and patience, moussaka is perfect for feeding a crowd
        or meal prepping for the week. The flavors actually improve after a day, making
        leftovers even more delicious.""",
        "cuisine": "Greek",
        "difficulty": "Intermediate",
        "time": "2 hours"
    },
    {
        "title": "Korean Bibimbap with Gochujang",
        "description": """A colorful Korean rice bowl that's as beautiful as it is delicious.
        Warm rice is topped with an array of seasoned vegetables—sautéed spinach, carrots,
        bean sprouts, mushrooms—along with marinated beef, a fried egg, and a generous
        dollop of spicy-sweet gochujang sauce. The name means 'mixed rice,' and the ritual
        of stirring everything together before eating is essential. Each ingredient is
        prepared separately, showcasing different cooking techniques and seasonings. The
        result is a harmonious bowl where every bite offers different flavors and textures.
        Customizable and nutritious, it's perfect for using up vegetables and experimenting
        with Korean flavors.""",
        "cuisine": "Korean",
        "difficulty": "Intermediate",
        "time": "1 hour"
    },
    {
        "title": "Mexican Carnitas Tacos",
        "description": """Authentic slow-cooked pork that's tender on the inside with
        crispy, caramelized edges. A pork shoulder is braised low and slow in its own
        fat with orange juice, garlic, and Mexican spices until it's so tender it falls
        apart with a fork. Then it's crisped under the broiler for textural contrast.
        Tucked into warm corn tortillas and topped with fresh cilantro, diced onions,
        lime, and your favorite salsa. These tacos are a weekend project worth every
        minute—the kind of food that brings everyone to the table. Serve with refried
        beans and Mexican rice for an authentic taqueria experience at home.""",
        "cuisine": "Mexican",
        "difficulty": "Easy",
        "time": "3.5 hours (mostly hands-off)"
    }
]

## Step 2: Implement Dense and Sparse Encoding

In [8]:
# Dense embeddings
encoder = SentenceTransformer("all-MiniLM-L6-v2")

# Global vocabulary - automatically extends as new texts are processed
global_vocabulary = {}

# Simple sparse encoding (BM25-style)
def create_sparse_vector(text):
    """Create sparse vector from text using term frequency"""
    from collections import Counter
    import re
    
    # Simple tokenization
    words = re.findall(r"\b\w+\b", text.lower())
    word_counts = Counter(words)
    
    # Convert to sparse vector format, extending vocabulary as needed
    indices = []
    values = []
    
    for word, count in word_counts.items():
        if word not in global_vocabulary:
            # Add new word to vocabulary with next available index
            global_vocabulary[word] = len(global_vocabulary)
        
        indices.append(global_vocabulary[word])
        values.append(float(count))
    
    return models.SparseVector(indices=indices, values=values)

# Upload hybrid data
points = []
for i, item in enumerate(my_dataset):
    dense_vector = encoder.encode(item["description"]).tolist()
    sparse_vector = create_sparse_vector(item["description"])
    
    points.append(models.PointStruct(
        id=i,
        vector={
            "dense": dense_vector, 
            "sparse": sparse_vector
        },
        payload=item
    ))

client.upload_points(collection_name=collection_name, points=points)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Step 3: Implement Hybrid Search with RRF

In [9]:
def hybrid_search_with_rrf(query_text, limit=10):
    """Perform hybrid search using Reciprocal Rank Fusion"""
    
    # Encode query for both dense and sparse
    query_dense = encoder.encode(query_text).tolist()
    query_sparse = create_sparse_vector(query_text)
    
    # Use Qdrant's built-in RRF
    response = client.query_points(
        collection_name=collection_name,
        prefetch=[
            models.Prefetch(
                query=query_dense,
                using="dense",
                limit=20
            ),
            models.Prefetch(
                query=query_sparse,
                using="sparse",
                limit=20
            )
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=limit
    )
    
    return response.points

# Test hybrid search
results = hybrid_search_with_rrf("your test query")
for i, point in enumerate(results, 1):
    print(f"{i}. {point.payload.get('title', 'No title')} (Score: {point.score:.3f})")

1. Thai Green Curry with Chicken (Score: 0.643)
2. Mexican Carnitas Tacos (Score: 0.583)
3. Korean Bibimbap with Gochujang (Score: 0.500)
4. Indian Butter Chicken (Murgh Makhani) (Score: 0.333)
5. Spanish Paella Valenciana (Score: 0.200)
6. Vietnamese Pho Bo (Beef Noodle Soup) (Score: 0.167)
7. Classic Chicken Caesar Salad (Score: 0.125)
8. Greek Moussaka (Score: 0.111)
9. Homemade Margherita Pizza (Score: 0.100)
10. Moroccan Lamb Tagine with Apricots (Score: 0.091)


## Step 4: Compare Search Approaches

In [10]:
def compare_search_methods(query_text):
    """Compare dense, sparse, and hybrid search results"""
    
    print(f"Query: '{query_text}'\n")
    
    # Dense-only search
    dense_results = client.query_points(
        collection_name=collection_name,
        query=encoder.encode(query_text).tolist(),
        using="dense",
        limit=5
    )
    
    # Sparse-only search  
    sparse_results = client.query_points(
        collection_name=collection_name,
        query=create_sparse_vector(query_text),
        using="sparse",
        limit=5
    )
    
    # Hybrid search
    hybrid_results = hybrid_search_with_rrf(query_text, limit=5)
    
    print("DENSE SEARCH:")
    for i, point in enumerate(dense_results.points, 1):
        print(f"  {i}. {point.payload.get('title', 'No title')} ({point.score:.3f})")
    
    print("\nSPARSE SEARCH:")
    for i, point in enumerate(sparse_results.points, 1):
        print(f"  {i}. {point.payload.get('title', 'No title')} ({point.score:.3f})")
    
    print("\nHYBRID SEARCH (RRF):")
    for i, point in enumerate(hybrid_results, 1):
        print(f"  {i}. {point.payload.get('title', 'No title')} ({point.score:.3f})")
    
    print("-" * 50)

# Test with different query types
test_queries = [
    "exact keyword match query",
    "semantic concept query", 
    "mixed keyword and concept query"
]

for query in test_queries:
    compare_search_methods(query)

Query: 'exact keyword match query'

DENSE SEARCH:
  1. Moroccan Lamb Tagine with Apricots (0.074)
  2. Vietnamese Pho Bo (Beef Noodle Soup) (0.070)
  3. Classic Chicken Caesar Salad (0.057)
  4. Miso-Glazed Salmon with Sesame Vegetables (0.055)
  5. Korean Bibimbap with Gochujang (0.041)

SPARSE SEARCH:

HYBRID SEARCH (RRF):
  1. Moroccan Lamb Tagine with Apricots (0.500)
  2. Vietnamese Pho Bo (Beef Noodle Soup) (0.333)
  3. Classic Chicken Caesar Salad (0.250)
  4. Miso-Glazed Salmon with Sesame Vegetables (0.200)
  5. Korean Bibimbap with Gochujang (0.167)
--------------------------------------------------
Query: 'semantic concept query'

DENSE SEARCH:
  1. Indian Butter Chicken (Murgh Makhani) (0.061)
  2. Thai Green Curry with Chicken (0.059)
  3. Classic Chicken Caesar Salad (0.055)
  4. Korean Bibimbap with Gochujang (0.052)
  5. Greek Moussaka (0.032)

SPARSE SEARCH:

HYBRID SEARCH (RRF):
  1. Indian Butter Chicken (Murgh Makhani) (0.500)
  2. Thai Green Curry with Chicken (0.3